In [ ]:
import torch
import torch.nn.functional as F
from torchvision import transforms as T
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import nrrd
import pandas as pd
import os
import sys

# sys.path.append("/mnt/raid/C1_ML_Analysis/source/autoencoder/src")
sys.path.append("/mnt/raid/C1_ML_Analysis/source/famli-ultra-sim/dl/")
sys.path.append("/mnt/raid/C1_ML_Analysis/source/famli-ultra-sim/dl/nets")

from nets import diffusion, spade, lotus, cut
from generative.inferers import DiffusionInferer
from generative.networks.nets import DiffusionModelUNet, AutoencoderKL, VQVAE, PatchDiscriminator, MultiScalePatchDiscriminator
from generative.networks.schedulers import DDPMScheduler

import monai
from monai.transforms import (    
    Compose,    
    RandFlip,
    RandRotate,
    CenterSpatialCrop,
    ScaleIntensityRange,
    RandAdjustContrast,
    RandGaussianNoise,
    RandGaussianSmooth
)

from loaders.ultrasound_dataset import USDatasetV2, LotusDataset
from transforms.ultrasound_transforms import LotusEvalTransforms, LotusTrainTransforms, RealUSTrainTransformsV2, RealUSEvalTransformsV2
from torchvision import transforms

In [ ]:
mount_point = '/mnt/raid/C1_ML_Analysis'
df = pd.read_parquet(os.path.join(mount_point, "CSV_files", "extract_frames_blind_sweeps_c1_30082022_wscores_test_w16_kmeans_sample_train.parquet"))
df = df.query('score > 0.999').reset_index(drop=True)

In [ ]:
mpl.rcParams['figure.figsize'] = (12, 8)

img_np, head = nrrd.read(os.path.join(mount_point, df['img_path'][0]), index_order="C")

img = torch.FloatTensor(img_np/255)

def input_T(input):
    # [0,1] -> [-1,+1]
    return 2*input-1
    
def output_T(input):
    # [-1,+1] -> [0,1]
    return (input+1)/2

def show(input):
    plt.imshow(output_T(input).clip(0,1), cmap='gray')
    
img_=input_T(img)
show(img_)


In [ ]:
#Defining the schedule for the diffusion process
num_timesteps=1000
betas=torch.linspace(1e-4,2e-2,num_timesteps)

alphas=1-betas
alphas_sqrt=alphas.sqrt()
alphas_cumprod=torch.cumprod(alphas,0)
alphas_cumprod_sqrt=alphas_cumprod.sqrt()

In [ ]:
def forward_step(t, condition_img, return_noise=False):
    """
        forward step: t-1 -> t
    """    
    assert t >= 0

    mean=alphas_sqrt[t]*condition_img    
    std=betas[t].sqrt()
      
    # sampling from N
    if not return_noise:
        return mean+std*torch.randn_like(img)
    else:
        noise=torch.randn_like(img)
        return mean+std*noise, noise
    
def forward_jump(t, condition_img, condition_idx=0, return_noise=False):
    """
        forward jump: 0 -> t
    """   
    assert t >= 0
    
    mean=alphas_cumprod_sqrt[t]*condition_img
    std=(1-alphas_cumprod[t]).sqrt()
      
    # sampling from N
    if not return_noise:
        return mean+std*torch.randn_like(img)
    else:
        noise=torch.randn_like(img)
        return mean+std*noise, noise

In [ ]:
N=5 # number of computed states between x_0 and x_T
M=4 # number of samples taken from each distribution

In [ ]:
plt.figure(figsize=(12,8))
for idx in range(N):
    t_step=int(idx*(num_timesteps/N))
    
    plt.subplot(N,1+M,1+(M+1)*idx)
    show(alphas_cumprod_sqrt[t_step]*img_)
    plt.title(r'$\mu_t=\sqrt{\bar{\alpha}_t}x_0$') if idx==0 else None
    plt.ylabel("t: {:.2f}".format(t_step/num_timesteps))
    plt.xticks([])
    plt.yticks([])
    
    for sample in range(M):
        x_t=forward_jump(t_step,img_)
        
        plt.subplot(N,1+M,2+(1+M)*idx+sample)
        show(x_t)        
        plt.axis('off')
        
plt.tight_layout()

In [ ]:
plt.figure(figsize=(12,8))
for idx in range(N):
    t_step=int(idx*(num_timesteps/N))
    prev_img=forward_jump(max([0,t_step-1]),img_) # directly go to prev state
    
    plt.subplot(N,1+M,1+(M+1)*idx)
    show(alphas_sqrt[t_step]*prev_img)
    plt.title(r'$\mu_t=\sqrt{1-\beta_t}x_{t-1}$') if idx==0 else None
    plt.ylabel("t: {:.2f}".format(t_step/num_timesteps))
    plt.xticks([])
    plt.yticks([])
    
    for sample in range(M):
        plt.subplot(N,1+M,2+(1+M)*idx+sample)
        x_t=forward_step(t_step,prev_img)
        show(x_t)        
        plt.axis('off')
plt.tight_layout()

In [ ]:
t_step=50

x_t,noise=forward_jump(t_step,img_,return_noise=True)

plt.subplot(1,3,1)
show(img_)
plt.title(r'$x_0$')
plt.axis('off')
plt.subplot(1,3,2)
show(x_t)
plt.title(r'$x_t$')
plt.axis('off')
plt.subplot(1,3,3)
show(noise)
plt.title(r'$\epsilon$')
plt.axis('off')

In [ ]:
x_0_pred=(x_t-(1-alphas_cumprod[t_step]).sqrt()*noise)/(alphas_cumprod_sqrt[t_step])

plt.subplot(1,3,1)
show(x_t)
plt.title('$x_t$ ($\ell_1$: {:.3f})'.format(F.l1_loss(x_t,img_)))
plt.axis('off')
plt.subplot(1,3,2)
show(x_0_pred)
plt.title('$x_0$ prediction ($\ell_1$: {:.3f})'.format(F.l1_loss(x_0_pred,img_)))
plt.axis('off') 
plt.subplot(1,3,3)
show(img_)
plt.title('$x_0$')
plt.axis('off')

In [ ]:
# estimate mean
mean_pred=x_0_pred*(alphas_cumprod_sqrt[t_step-1]*betas[t_step])/(1-alphas_cumprod[t_step]) + x_t*(alphas_sqrt[t_step]*(1-alphas_cumprod[t_step-1]))/(1-alphas_cumprod[t_step])

# let's compare it to ground truth mean of the previous step (requires knowledge of x_0)
mean_gt=alphas_cumprod_sqrt[t_step-1]*img_

In [ ]:
plt.subplot(1,3,1)
show(x_t)
plt.title('$x_t$   ($\ell_1$: {:.3f})'.format(F.l1_loss(x_t,img_)))
plt.subplot(1,3,2)
show(mean_pred)
plt.title(r'$\tilde{\mu}_{t-1}$' + '  ($\ell_1$: {:.3f})'.format(F.l1_loss(mean_pred,img_)))
plt.subplot(1,3,3)
show(mean_gt)
plt.title(r'$\mu_{t-1}$' + '  ($\ell_1$: {:.3f})'.format(F.l1_loss(mean_gt,img_)))

In [ ]:
def reverse_step(epsilon, x_t, t_step, return_noise=False):
    
    # estimate x_0 based on epsilon
    x_0_pred=(x_t-(1-alphas_cumprod[t_step]).sqrt()*epsilon)/(alphas_cumprod_sqrt[t_step])
    if t_step==0:
        sample=x_0_pred
        noise=torch.zeros_like(x_0_pred)
    else:
        # estimate mean
        mean_pred=x_0_pred*(alphas_cumprod_sqrt[t_step-1]*betas[t_step])/(1-alphas_cumprod[t_step]) + x_t*(alphas_sqrt[t_step]*(1-alphas_cumprod[t_step-1]))/(1-alphas_cumprod[t_step])

        # compute variance
        beta_pred=betas[t_step].sqrt() if t_step != 0 else 0

        sample=mean_pred+beta_pred*torch.randn_like(x_t)
        # this noise is only computed for simulation purposes (since x_0_pred is not known normally)
        noise=(sample-x_0_pred*alphas_cumprod_sqrt[t_step-1])/(1-alphas_cumprod[t_step-1]).sqrt()
    if return_noise:
        return sample, noise
    else:
        return sample

In [ ]:
model_fn = "/mnt/raid/C1_ML_Analysis/train_output/diffusion/extract_frames_Dataset_C_masked_resampled_256_spc075_wscores_meta_BPD01_MACFL025-7mo-9mo/v0.1/last.ckpt"

NN = getattr(diffusion, "DDPMPL")
model = NN.load_from_checkpoint(model_fn)
model.eval()
model.cuda()

In [ ]:
scheduler = DDPMScheduler(num_train_timesteps=1000)
inferer = DiffusionInferer(scheduler=scheduler)


In [ ]:
noise = torch.randn((1, 1, 128, 128))
noise = noise.cuda()
scheduler.set_timesteps(num_inference_steps=100)


image, intermediates = inferer.sample(
    input_noise=noise, diffusion_model=model.model, scheduler=scheduler, save_intermediates=True, intermediate_steps=100
)

chain = torch.cat(intermediates, dim=-1)

plt.figure(figsize=(12,8))
plt.style.use("default")
plt.imshow(chain[0, 0].cpu(), vmin=0, vmax=1)
plt.tight_layout()
plt.axis("off")
plt.show()

plt.style.use("default")
plt.imshow(image[0].cpu().permute(1,2,0), vmin=0, vmax=1)
plt.tight_layout()
plt.axis("off")
plt.show()

In [ ]:
# USDatasetV2(self.df_test, self.mount_point, img_column=self.img_column, class_column=self.class_column, ga_column=self.ga_column, scalar_column=self.scalar_column, transform=self.test_transform)

model_fn = '/mnt/raid/C1_ML_Analysis/train_output/ultra-sim/rendering_spade_linear/v0.2/epoch=5-val_loss=6.72.ckpt'

model_sp = spade.SPADELotus.load_from_checkpoint(model_fn)
model_sp.eval()
model_sp.cuda()

df_test = pd.read_csv('/mnt/raid/C1_ML_Analysis/source/blender/simulated_data_export/CSV_files/label_diffusor_v2_test.csv')
mount_point = '/mnt/raid/C1_ML_Analysis/source/blender/simulated_data_export' 

test_ds = monai.data.Dataset(data=LotusDataset(df_test, mount_point), transform=LotusEvalTransforms())

In [ ]:
sp_out = model_sp(test_ds[0]['seg'][None,:].cuda())[0]
plt.style.use("default")
plt.imshow(sp_out[0].permute(1,2,0).detach().cpu().numpy(), vmin=0, vmax=1)
plt.tight_layout()
plt.axis("off")
plt.show()

In [ ]:




# noise = torch.randn((1, 1, 128, 128))
# noise = noise.cuda()
noise = transforms.Resize(128)(sp_out)
# scheduler.set_timesteps(num_inference_steps=100)


image, intermediates = inferer.sample(
    input_noise=noise, diffusion_model=model.model, scheduler=scheduler, save_intermediates=True, intermediate_steps=5
)

chain = torch.cat(intermediates, dim=-1)

plt.figure(figsize=(12,8))
plt.style.use("default")
plt.imshow(chain[0, 0].cpu(), vmin=0, vmax=1)
plt.tight_layout()
plt.axis("off")
plt.show()

plt.style.use("default")
plt.imshow(image[0].cpu().permute(1,2,0), vmin=0, vmax=1)
plt.tight_layout()
plt.axis("off")
plt.show()

In [ ]:
noise = torch.randn((1, 1, 128, 128))
noise = noise.cuda()
noisy_image = scheduler.add_noise(original_samples=transforms.Resize(128)(sp_out), noise=noise, timesteps=torch.tensor(30))
plt.style.use("default")
plt.imshow(noisy_image[0].cpu().detach().permute(1,2,0).numpy(), vmin=0, vmax=1)
plt.tight_layout()
plt.axis("off")
plt.show()

In [ ]:
# noise = transforms.Resize(128)(sp_out)
scheduler.set_timesteps(num_inference_steps=30)


image, intermediates = inferer.sample(
    input_noise=noisy_image, diffusion_model=model.model, scheduler=scheduler, save_intermediates=True, intermediate_steps=1
)

chain = torch.cat(intermediates, dim=-1)

plt.figure(figsize=(12,8))
plt.style.use("default")
plt.imshow(chain[0, 0].cpu(), vmin=0, vmax=1)
plt.tight_layout()
plt.axis("off")
plt.show()

plt.style.use("default")
plt.imshow(image[0].cpu().permute(1,2,0), vmin=0, vmax=1)
plt.tight_layout()
plt.axis("off")
plt.show()

In [ ]:
from torchvision.transforms import v2
# torchvision.transforms.v2.RandomErasing

In [ ]:
img_erase = v2.RandomErasing()(noisy_image)
plt.style.use("default")
plt.imshow(img_erase[0].cpu().detach().permute(1,2,0).numpy(), vmin=0, vmax=1)
plt.tight_layout()
plt.axis("off")
plt.show()

In [ ]:
# model_fn = "/mnt/raid/C1_ML_Analysis/train_output/diffusion/extract_frames_Dataset_C_masked_resampled_256_spc075_wscores_meta_BPDACFLL1MR0C1-7mo-9mo/v0.1/last.ckpt"
model_fn = '/mnt/raid/C1_ML_Analysis/train_output/diffusionAE/extract_frames_Dataset_C_masked_resampled_256_spc075_wscores_meta_BPD01_MACFL025-7mo-9mo/v0.4/epoch=72-val_loss=0.01.ckpt'
model_ae = diffusion.AutoEncoderKL.load_from_checkpoint(model_fn)
model_ae.cuda()
model_ae.eval()

In [ ]:
plt.style.use("default")
plt.imshow(model_ae(transforms.Resize(128)(sp_out.cuda()))[0][0].cpu().detach().permute(1,2,0).numpy(), vmin=0, vmax=1)
plt.tight_layout()
plt.axis("off")
plt.show()

In [ ]:
model_ae.autoencoderkl.encoder(transforms.Resize(128)(sp_out.cuda())).shape

In [ ]:
discriminator = MultiScalePatchDiscriminator(
    num_d=2,
    num_layers_d=3,
    spatial_dims=2,
    num_channels=8,
    in_channels=1,
    out_channels=8,
    minimum_size_im=256,
    norm="INSTANCE",
    kernel_size=3,
)

In [ ]:
def feature_loss(input_features_disc_fake, input_features_disc_real, lambda_feat):
    criterion = torch.nn.L1Loss()
    num_D = len(input_features_disc_fake)
    GAN_Feat_loss = torch.zeros(1)
    for i in range(num_D):  # for each discriminator
        num_intermediate_outputs = len(input_features_disc_fake[i])
        for j in range(num_intermediate_outputs):  # for each layer output
            unweighted_loss = criterion(input_features_disc_fake[i][j], input_features_disc_real[i][j].detach())
            GAN_Feat_loss += unweighted_loss * lambda_feat / num_D
    return GAN_Feat_loss

In [ ]:
fake = torch.rand(1, 1, 256, 256)
disc_fakes, features_fakes = discriminator(fake)
# loss_g = adversarial_loss(disc_fakes, target_is_real=True, for_discriminator=False)

real = torch.rand(1, 1, 256, 256)
disc_reals, features_reals = discriminator(real)
loss_feat = feature_loss(features_fakes, features_reals, lambda_feat=0.1)


In [ ]:
#for discriminator

# disc_fakes, _ = discriminator(torch.cat([out, label], 1))
# loss_d_r = adversarial_loss(disc_reals, target_is_real=True, for_discriminator=True)
# loss_g_f = adversarial_loss(disc_fakes, target_is_real=False, for_discriminator=True)
# optimizer_D.zero_grad()
# loss_d = loss_d_r + loss_g_f
# loss_d.backward()

In [ ]:
torch.argsort(1 - torch.tensor([-1, 0, 1, 0.1]))

In [ ]:
model_fn = '/mnt/raid/C1_ML_Analysis/train_output/ultra-sim/rendering_cut_linear/v0.2/epoch=39-val_loss=3.66.ckpt'

model_cu = cut.CutLinear.load_from_checkpoint(model_fn)
model_cu.eval()
model_cu.cuda()

In [ ]:
sp_out = model_cu(test_ds[0]['seg'][None,:].cuda())
plt.style.use("default")
plt.imshow(sp_out[0].permute(1,2,0).detach().cpu().numpy(), vmin=0, vmax=1)
plt.tight_layout()
plt.axis("off")
plt.show()

In [ ]:
plt.style.use("default")
plt.imshow(model_ae(sp_out.cuda())[0][0].cpu().detach().permute(1,2,0).numpy(), vmin=0, vmax=1)
plt.tight_layout()
plt.axis("off")
plt.show()

In [ ]:

class MergedCut(torch.nn.Module):
    def __init__(self):
        super().__init__()

        model_fn = '/mnt/raid/C1_ML_Analysis/train_output/ultra-sim/rendering_cut_linear/v0.2/epoch=39-val_loss=3.66.ckpt'

        self.cut = cut.CutLinear.load_from_checkpoint(model_fn).G
        self.cut.cuda()

        model_fn = '/mnt/raid/C1_ML_Analysis/train_output/diffusionAE/extract_frames_Dataset_C_masked_resampled_256_spc075_wscores_meta_BPD01_MACFL025-7mo-9mo/v0.4/epoch=72-val_loss=0.01.ckpt'
        self.model_ae = diffusion.AutoEncoderKL.load_from_checkpoint(model_fn).autoencoderkl
        self.model_ae.cuda()

    def forward(self, X):
        X = self.cut(X)
        return self.model_ae(X)[0]
        

In [ ]:
# us_simu = MergedCut()

In [ ]:
# us_simu.eval()
# us_simu = us_simu.cuda()

In [ ]:

# simu_out = model_cu.transform_us(model_cu.USR(test_ds[0]['seg'][None,:].cuda()))

# simu_out = us_simu(simu_out)*model_cu.transform_us(model_cu.USR.mask_fan).cuda()
# plt.style.use("default")
# plt.imshow(simu_out[0].permute(1,2,0).detach().cpu().numpy(), vmin=0, vmax=1)
# plt.tight_layout()
# plt.axis("off")
# plt.show()

In [ ]:
# us_simu_traced = torch.jit.trace(us_simu, torch.rand(1, 1, 256, 256).cuda())

In [ ]:
# simu_out = model_cu.transform_us(model_cu.USR(test_ds[0]['seg'][None,:].cuda()))

# simu_out = us_simu_traced(simu_out)*model_cu.transform_us(model_cu.USR.mask_fan).cuda()
# plt.style.use("default")
# plt.imshow(simu_out[0].permute(1,2,0).detach().cpu().numpy(), vmin=0, vmax=1)
# plt.tight_layout()
# plt.axis("off")
# plt.show()

In [ ]:
# us_simu_traced.save('/mnt/famli_netapp_shared/C1_ML_Analysis/src/famli-ultra-sim/trained_models/cut_ae.pt')

In [ ]:
us_simu_traced = torch.jit.load('/mnt/famli_netapp_shared/C1_ML_Analysis/src/famli-ultra-sim/trained_models/cut_linear_ae.pt')

simu_out = model_cu.transform_us(model_cu.USR(test_ds[10]['seg'][None,:].cuda()))

simu_out = us_simu_traced(simu_out)*model_cu.transform_us(model_cu.USR.mask_fan).cuda()
plt.style.use("default")
plt.imshow(simu_out[0].permute(1,2,0).detach().cpu().numpy(), vmin=0, vmax=1)
plt.tight_layout()
plt.axis("off")
plt.show()

In [ ]:
class MergedCut2(torch.nn.Module):
    def __init__(self):
        super().__init__()

        model_fn = '/mnt/raid/C1_ML_Analysis/train_output/ultra-sim/rendering_cut/v0.11/epoch=19-val_loss=3.30.ckpt'

        self.cut = cut.Cut.load_from_checkpoint(model_fn).G
        self.cut.cuda()

        model_fn = '/mnt/raid/C1_ML_Analysis/train_output/diffusionAE/extract_frames_Dataset_C_masked_resampled_256_spc075_wscores_meta_BPD01_MACFL025-7mo-9mo/v0.4/epoch=72-val_loss=0.01.ckpt'
        self.model_ae = diffusion.AutoEncoderKL.load_from_checkpoint(model_fn).autoencoderkl
        self.model_ae.cuda()

    def forward(self, X):
        X = self.cut(X)
        return self.model_ae(X)[0]

In [ ]:
# us_simu = MergedCut2()
# us_simu_traced = torch.jit.trace(us_simu, torch.rand(1, 1, 256, 256).cuda())
# us_simu_traced.save('/mnt/famli_netapp_shared/C1_ML_Analysis/src/famli-ultra-sim/trained_models/cut_v0.11-ae_v0.4.pt')

In [ ]:
us_simu_traced = torch.jit.load('/mnt/famli_netapp_shared/C1_ML_Analysis/src/famli-ultra-sim/trained_models/cut_v0.11-ae_v0.4.pt')

simu_out = model_cu.transform_us(model_cu.USR(test_ds[0]['seg'][None,:].cuda()))

simu_out = us_simu_traced(simu_out)*model_cu.transform_us(model_cu.USR.mask_fan).cuda()
plt.style.use("default")
plt.imshow(simu_out[0].permute(1,2,0).detach().cpu().numpy(), vmin=0, vmax=1)
plt.tight_layout()
plt.axis("off")
plt.show()

In [ ]:
class MergedCut3(torch.nn.Module):
    def __init__(self):
        super().__init__()

        USR = torch.jit.load("/mnt/famli_netapp_shared/C1_ML_Analysis/src/famli-ultra-sim/trained_models/cut_v0.12-ae_v0.4_USR.pt")
        G = torch.jit.load("/mnt/famli_netapp_shared/C1_ML_Analysis/src/famli-ultra-sim/trained_models/cut_v0.12-ae_v0.4_G.pt")
        AE = torch.jit.load("/mnt/famli_netapp_shared/C1_ML_Analysis/src/famli-ultra-sim/trained_models/cut_v0.12-ae_v0.4_AE.pt")

        self.register_module('USR', USR)
        self.register_module('G', G)
        self.register_module('AE', AE)

        self.transform_us = T.Compose([T.Pad((0, 80, 0, 0)), T.CenterCrop(256)])

    def init_grids(self, w, h, center_x, center_y, r1, r2, theta):
        grid = self.compute_grid(w, h, center_x, center_y, r1, r2, theta)
        inverse_grid, mask = self.compute_grid_inverse(grid)
        grid = self.normalize_grid(grid)
        inverse_grid = self.normalize_grid(inverse_grid)
        
        return  grid, inverse_grid, mask

    def compute_grid(self, w, h, center_x, center_y, r1, r2, theta):

        # Convert inputs to tensors
        angles = torch.linspace(-theta, theta, w)  # Angles from -theta to theta
        radii = torch.linspace(r1, r2, h)  # Linear space of radii

        # Calculate sin and cos for all angles (broadcasting)
        sin_angles = torch.sin(angles)
        cos_angles = torch.cos(angles)

        # Initialize the grid for intersection points
        # Shape of grid: (h, w, 2) where 2 represents (x, y) coordinates
        grid = torch.zeros(h, w, 2)

        # Calculate intersections for each radius and angle
        for i, radius in enumerate(radii):
            x = (center_x + radius * sin_angles) # x coordinates for all angles at this radius
            y = (center_y + radius * cos_angles) # y coordinates for all angles at this radius

            grid[i] = torch.stack((x, y), dim=1)  # Update grid with coordinates

        return grid
        

    def compute_grid_inverse(self, grid):

        h, w, _ = grid.shape  # grid dimensions
        inverse_grid = torch.zeros(h, w, 2)  # Initialize inverse grid
        mask = torch.zeros(1, h, w)  # Initialize mask

        # Iterate through each point in the grid
        for j in range(h):
            for i in range(w):
                # Extract the polar coordinates (represented in the grid)
                xi, yi = torch.round(grid[j, i]).to(torch.long)

                # Place the Cartesian coordinates in the inverse grid
                if 0 <= xi and xi < w and 0 <= yi and yi < h:
                    inverse_grid[yi, xi] = torch.tensor([i, j])
                    mask[0, yi, xi] = 1
        return inverse_grid, self.morphology_close(mask.unsqueeze(0)).squeeze(0)

    def normalize_grid(self, grid):
        h, w, _ = grid.shape  # grid dimensions
        grid = grid / torch.tensor([h, w]) * 2.0 - 1.0
        return grid

    def dilate(self, x, kernel_size = 3):
        kernel = torch.ones((1, 1, kernel_size, kernel_size), dtype=torch.float32)

        # Apply convolution to simulate dilation
        # We use padding=1 to ensure the output size is the same as the input size
        output = F.conv2d(x, kernel, padding=1)

        # Apply a threshold to get a binary output
        dilated_image = (output > 0).float()

        return dilated_image
    
    def erode(self, x, kernel_size = 3):
        # Step 2: Erosion
        # For erosion, invert the image and kernel, apply dilation, then invert the output
        x = 1 - x
        inverted_kernel = torch.ones((1, 1, kernel_size, kernel_size), dtype=torch.float32) # Same kernel as for dilation

        # Apply convolution (dilation on inverted image) with padding to maintain size
        eroded_output_inverted = F.conv2d(x, inverted_kernel, padding=1)

        # Invert the result to get the final eroded (closing) result
        eroded_image = 1 - (eroded_output_inverted > 0).float()

        return eroded_image

    def morphology_close(self, x, kernel_size=3):
        return self.erode(self.dilate(x, kernel_size), kernel_size)

    def forward(self, X, grid, inverse_grid, mask_fan):
        
        X = self.USR(X, grid, inverse_grid, mask_fan)
        X = self.transform_us(X)
        X = self.G(X)
        return self.AE(X)[0]*self.transform_us(mask_fan)

In [ ]:
us_cut_v012 = MergedCut3().cuda()

grid, inverse_grid, mask_fan = us_cut_v012.init_grids(256, 256, 128.0, -30.0, 20.0, 215.0, 0.7853981633974483)

repeats = [1,]*4

grid = grid.repeat(repeats)
inverse_grid = inverse_grid.repeat(repeats)
mask_fan = mask_fan.repeat(repeats)



In [ ]:
x = us_cut_v012(test_ds[10]['seg'][None,:].cuda(), grid.cuda(), inverse_grid.cuda(), mask_fan.cuda())
plt.style.use("default")
plt.imshow(x[0].permute(1,2,0).detach().cpu().numpy(), vmin=0, vmax=1)
plt.tight_layout()
plt.axis("off")
plt.show()